# Term SOFR Approximation Calculator

Approximates forward-looking Term SOFR rates **(1M, 3M, 6M, 12M)** by bootstrapping implied rates from CME SR1 (1-month) and SR3 (3-month) SOFR futures prices sourced from Yahoo Finance via `yfinance`.

---

> **DISCLAIMER:** This produces an **approximation** of Term SOFR — not the official CME fixing.  
> CME uses intraday VWAP; this notebook uses end-of-day settlement prices.  
> Typical deviation: 0.5–3 basis points.  
> Suitable for internal modelling, budgeting, and underwriting.  
> **NOT suitable as a contractual reference rate.**

## 1 — Install Dependencies

In [ ]:
!pip install yfinance pandas numpy requests --quiet

## 2 — Imports

In [ ]:
import csv
import calendar
from datetime import date, timedelta
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import requests
import yfinance as yf

## 3 — Configuration

Edit the variables in this cell to control what the notebook calculates.

| Variable | Description |
|---|---|
| `HISTORY_DAYS` | `None` = today only; set to an integer (e.g. `90`) for a historical run |
| `OUTPUT_PATH` | Path for the exported CSV (auto-named if `None`) |

In [ ]:
# ── USER CONFIG ───────────────────────────────────────────────
HISTORY_DAYS: Optional[int] = None   # e.g. 90 for last 90 days; None for today only
OUTPUT_PATH:  Optional[str] = None   # e.g. "rates.csv"; None = auto-named
# ─────────────────────────────────────────────────────────────

IS_HISTORICAL = HISTORY_DAYS is not None

def _default_output(historical: bool) -> Path:
    tag   = "historical" if historical else "current"
    stamp = date.today().strftime("%Y%m%d")
    return Path(f"term_sofr_{tag}_{stamp}.csv")

output_path = Path(OUTPUT_PATH) if OUTPUT_PATH else _default_output(IS_HISTORICAL)
print(f"Mode     : {'Historical (' + str(HISTORY_DAYS) + ' days)' if IS_HISTORICAL else 'Current (today only)'}")
print(f"Output   : {output_path}")

## 4 — Contract Calendar Helpers

In [ ]:
# CME month codes: F=Jan G=Feb H=Mar J=Apr K=May M=Jun N=Jul Q=Aug U=Sep V=Oct X=Nov Z=Dec
MONTH_CODES = {
    1:"F", 2:"G", 3:"H", 4:"J", 5:"K", 6:"M",
    7:"N", 8:"Q", 9:"U", 10:"V", 11:"X", 12:"Z"
}

SR3_MONTHS = {3, 6, 9, 12}  # SR3 quarterly expiries: Mar/Jun/Sep/Dec


def get_sr1_tickers(n: int = 14, ref_date: Optional[date] = None) -> list[dict]:
    """Next `n` SR1 (1-month SOFR) futures tickers from ref_date."""
    ref = ref_date or date.today()
    tickers = []
    year, month = ref.year, ref.month
    for _ in range(n):
        code = MONTH_CODES[month]
        yy   = str(year)[-2:]
        tickers.append({
            "ticker":      f"SR1{code}{yy}.CME",
            "year":        year,
            "month":       month,
            "expiry_date": date(year, month, 1),
            "type":        "SR1",
        })
        month += 1
        if month > 12:
            month = 1
            year += 1
    return tickers


def get_sr3_tickers(n: int = 6, ref_date: Optional[date] = None) -> list[dict]:
    """Next `n` SR3 (3-month SOFR) quarterly futures tickers from ref_date."""
    ref = ref_date or date.today()
    tickers = []
    year, month = ref.year, ref.month
    while month not in SR3_MONTHS:
        month += 1
        if month > 12:
            month = 1
            year += 1
    for _ in range(n):
        code = MONTH_CODES[month]
        yy   = str(year)[-2:]
        tickers.append({
            "ticker":      f"SR3{code}{yy}.CME",
            "year":        year,
            "month":       month,
            "expiry_date": date(year, month, 1),
            "type":        "SR3",
        })
        month += 3
        if month > 12:
            month -= 12
            year += 1
    return tickers

## 5 — Fetch Futures Prices

In [ ]:
def fetch_prices(contracts: list[dict], period: str = "5d") -> dict[str, float]:
    """Fetch latest closing prices for contracts from Yahoo Finance."""
    tickers_list = [c["ticker"] for c in contracts]
    prices = {}
    print(f"  Fetching {len(tickers_list)} contracts from Yahoo Finance...")
    try:
        raw = yf.download(tickers_list, period=period, progress=False, auto_adjust=True)
        if raw.empty:
            print("  WARNING: No data returned.")
            return prices
        close = raw["Close"] if "Close" in raw.columns else raw
        for ticker in tickers_list:
            try:
                series = close[ticker].dropna() if ticker in close.columns else close.dropna()
                if not series.empty:
                    prices[ticker] = float(series.iloc[-1])
            except Exception:
                pass
    except Exception as e:
        print(f"  ERROR fetching prices: {e}")
    valid = {k: v for k, v in prices.items() if v > 50}
    print(f"  Retrieved {len(valid)}/{len(tickers_list)} valid contract prices.")
    return valid


def fetch_prices_historical(
    contracts: list[dict],
    start: str,
    end: str,
) -> pd.DataFrame:
    """Fetch daily closing prices for all contracts over a date range."""
    tickers_list = [c["ticker"] for c in contracts]
    print(f"  Fetching historical prices for {len(tickers_list)} contracts ({start} to {end})...")
    try:
        raw = yf.download(tickers_list, start=start, end=end, progress=False, auto_adjust=True)
        if raw.empty:
            return pd.DataFrame()
        close = raw["Close"] if "Close" in raw.columns else raw
        close = close.loc[:, (close > 50).any()]
        return close
    except Exception as e:
        print(f"  ERROR: {e}")
        return pd.DataFrame()

## 6 — Overnight SOFR (NY Fed)

In [ ]:
def fetch_overnight_sofr() -> Optional[float]:
    """Fetch latest overnight SOFR from the NY Fed public API."""
    try:
        url  = "https://markets.newyorkfed.org/api/rates/sofr/last/1.json"
        r    = requests.get(url, timeout=10)
        r.raise_for_status()
        rate = r.json()["refRates"][0]["percentRate"]
        print(f"  Overnight SOFR (NY Fed): {rate:.4f}%")
        return float(rate)
    except Exception as e:
        print(f"  WARNING: Could not fetch overnight SOFR — {e}")
        return None

## 7 — Core Calculation

In [ ]:
def futures_price_to_rate(price: float) -> float:
    return 100.0 - price


def days_in_month(year: int, month: int) -> int:
    return calendar.monthrange(year, month)[1]


def build_implied_curve(
    sr1_contracts: list[dict],
    sr3_contracts: list[dict],
    prices: dict[str, float],
    overnight_sofr: Optional[float],
    ref_date: date,
) -> list[dict]:
    """Bootstrap a daily forward rate curve from SR1 futures prices."""
    curve = []
    for contract in sr1_contracts:
        ticker = contract["ticker"]
        if ticker not in prices:
            continue
        rate   = futures_price_to_rate(prices[ticker])
        year, month = contract["year"], contract["month"]
        n_days = days_in_month(year, month)
        d = date(year, month, 1)
        end = date(year, month, n_days)
        while d <= end:
            if d >= ref_date:
                curve.append({"date": d.isoformat(), "implied_rate": rate, "source": ticker})
            d += timedelta(days=1)
    curve.sort(key=lambda x: x["date"])
    return curve


def compound_rate(curve_slice: list[dict], tenor_days: int) -> Optional[float]:
    """Compound daily implied rates over tenor_days using Act/360 SOFR convention."""
    if len(curve_slice) < tenor_days:
        return None
    compounded = 1.0
    for point in curve_slice[:tenor_days]:
        compounded *= (1 + (point["implied_rate"] / 100.0) * (1 / 360))
    return round((compounded - 1) * (360 / tenor_days) * 100, 5)


TENORS = {
    "1-Month Term SOFR":  30,
    "3-Month Term SOFR":  90,
    "6-Month Term SOFR":  180,
    "12-Month Term SOFR": 360,
}


def calculate_term_sofr(curve: list[dict], ref_date: date) -> dict[str, Optional[float]]:
    forward = [p for p in curve if p["date"] >= ref_date.isoformat()]
    return {label: compound_rate(forward, days) for label, days in TENORS.items()}

## 8 — Output Helpers

In [ ]:
def results_to_dataframe(results: dict, ref_date: date, overnight: Optional[float]) -> pd.DataFrame:
    rows = []
    if overnight:
        rows.append({"Date": ref_date.isoformat(), "Tenor": "Overnight",
                     "Rate (%)": round(overnight, 5), "Type": "Overnight SOFR",
                     "Source": "NY Fed API", "Note": "Official fixing"})
    for label, rate in results.items():
        rows.append({
            "Date":     ref_date.isoformat(),
            "Tenor":    label.replace(" Term SOFR", "").replace("-", "").strip(),
            "Rate (%)": round(rate, 5) if rate is not None else None,
            "Type":     "Term SOFR (Approximated)",
            "Source":   "CME SR1 Futures via Yahoo Finance",
            "Note":     "Bootstrapped from EOD settlement — not official CME fixing",
        })
    return pd.DataFrame(rows)


def write_csv(df: pd.DataFrame, path: Path) -> None:
    if df.empty:
        print("No data to export.")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Exported {len(df)} rows to {path.resolve()}")

## 9 — Run: Current Mode

Calculates implied Term SOFR rates as of **today**. Skipped automatically when `HISTORY_DAYS` is set.

In [ ]:
if not IS_HISTORICAL:
    ref_date = date.today()
    print(f"\nCurrent mode: calculating implied Term SOFR as of {ref_date}\n")

    sr1_contracts = get_sr1_tickers(n=14, ref_date=ref_date)
    sr3_contracts = get_sr3_tickers(n=6,  ref_date=ref_date)

    prices    = fetch_prices(sr1_contracts + sr3_contracts)
    overnight = fetch_overnight_sofr()

    if not prices:
        print("ERROR: No futures prices retrieved. Check internet connection.")
    else:
        curve   = build_implied_curve(sr1_contracts, sr3_contracts, prices, overnight, ref_date)
        results = calculate_term_sofr(curve, ref_date)

        df_current = results_to_dataframe(results, ref_date, overnight)
        display(df_current)
        write_csv(df_current, output_path)
else:
    print("Historical mode selected — skipping current-mode cell.")

## 10 — Run: Historical Mode

Calculates implied Term SOFR rates over the last `HISTORY_DAYS` calendar days. Skipped when `HISTORY_DAYS` is `None`.

In [ ]:
if IS_HISTORICAL:
    end_date   = date.today()
    start_date = end_date - timedelta(days=HISTORY_DAYS)
    print(f"\nHistorical mode: {start_date} to {end_date} ({HISTORY_DAYS} days)\n")

    sr1_contracts = get_sr1_tickers(n=16)
    sr3_contracts = get_sr3_tickers(n=8)
    all_contracts = sr1_contracts + sr3_contracts

    price_df = fetch_prices_historical(
        all_contracts,
        start=start_date.isoformat(),
        end=(end_date + timedelta(days=1)).isoformat(),
    )

    if price_df.empty:
        print("ERROR: No historical price data retrieved.")
    else:
        history_rows = []
        for ts, row in price_df.iterrows():
            ref = ts.date() if hasattr(ts, "date") else date.fromisoformat(str(ts)[:10])
            day_prices = {col: float(val) for col, val in row.items()
                          if pd.notna(val) and float(val) > 50}
            if not day_prices:
                continue
            curve = build_implied_curve(sr1_contracts, sr3_contracts, day_prices, None, ref)
            if not curve:
                continue
            for label, rate in calculate_term_sofr(curve, ref).items():
                if rate is not None:
                    history_rows.append({
                        "Date":     ref.isoformat(),
                        "Tenor":    label.replace(" Term SOFR", "").replace("-", "").strip(),
                        "Rate (%)": round(rate, 5),
                        "Type":     "Term SOFR (Approximated)",
                        "Source":   "CME SR1 Futures via Yahoo Finance",
                        "Note":     "Bootstrapped from EOD settlement — not official CME fixing",
                    })

        df_history = pd.DataFrame(history_rows)
        if not df_history.empty:
            print(f"\nCalculated {len(df_history)} rate observations across {df_history['Date'].nunique()} trading days.")
            print(f"\nLatest available rates ({df_history['Date'].max()}):")
            display(df_history[df_history["Date"] == df_history["Date"].max()][["Tenor", "Rate (%)"]].reset_index(drop=True))
            write_csv(df_history, output_path)
        else:
            print("No rates could be calculated from available data.")
else:
    print("Current mode selected — skipping historical-mode cell.")